# E006 — Image flagship: PCA + LDA (Fisherfaces)

**Why Fisherfaces over plain PCA (E004)?**

PCA maximizes total variance — it finds directions that capture the most
information about *all* faces. But we don't care about all faces: we care
about separating **one specific target** from everyone else.

LDA (Linear Discriminant Analysis) directly maximizes:
```
J(w) = (w^T S_B w) / (w^T S_W w)     # between-class / within-class scatter
```
This gives the direction where the two classes are most separated relative
to their internal spread. For binary classification there is exactly **1**
such direction — the Fisherface.

**Why PCA first?**  
LDA requires inverting the within-class scatter matrix S_W. With 6400 features
and only ~212 training samples, S_W is rank-deficient. PCA first reduces to a
subspace where S_W is invertible.

**Why Ledoit-Wolf shrinkage?**  
Even after PCA, with ~20 target samples per fold, the target within-class
covariance estimate is noisy. Ledoit-Wolf (`shrinkage='auto'`) regularizes
the covariance estimator toward a scaled identity — reducing estimation error
at the cost of some bias. This is the statistically correct approach for
small-sample LDA, not a workaround.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

In [ ]:
def find_png(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".png")
        if p.exists():
            return p
    raise FileNotFoundError(stem)

def load_image(png_path: Path) -> np.ndarray:
    img = np.array(Image.open(png_path).convert("RGB"), dtype=np.float32)
    return img.mean(axis=2).flatten()  # grayscale → flatten → (6400,)

def load_images(df: pd.DataFrame, data_dir: Path) -> np.ndarray:
    return np.stack([load_image(find_png(row["stem"], data_dir)) for _, row in df.iterrows()])

# Sanity check
X_all = load_images(manifest, DATA)
print(f"Feature matrix: {X_all.shape}  (samples × pixels)")

## 1. Fisherface visualization (full dataset — for insight only)

We fit PCA+LDA on the full dataset here purely for visualization.
**The actual CV uses only train-fold data for fitting** — this is just
to understand what the algorithm is doing.

The single LDA direction (the Fisherface) is the vector in face-space
that maximally separates Ondra from everyone else.

In [ ]:
# Fit on full data for visualization only
scaler_viz = StandardScaler()
X_s = scaler_viz.fit_transform(X_all)

pca_viz = PCA(n_components=100, random_state=67)
X_pca = pca_viz.fit_transform(X_s)

lda_viz = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
lda_viz.fit(X_pca, y_all)

# The Fisherface: project LDA direction back to pixel space
# lda_viz.coef_ is (1, n_pca_components), pca_viz.components_ is (n_pca, 6400)
fisherface = (lda_viz.coef_ @ pca_viz.components_).reshape(80, 80)

# Mean faces per class
mean_target    = X_all[y_all == 1].mean(axis=0).reshape(80, 80)
mean_nontarget = X_all[y_all == 0].mean(axis=0).reshape(80, 80)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(mean_target, cmap="gray")
axes[0].set_title("Mean face — target (m431)", color=COLORS["target"])
axes[0].axis("off")

axes[1].imshow(mean_nontarget, cmap="gray")
axes[1].set_title("Mean face — non-target", color=COLORS["nontarget"])
axes[1].axis("off")

axes[2].imshow(fisherface, cmap="RdBu_r")
axes[2].set_title("Fisherface (LDA direction)\nred=target, blue=non-target")
axes[2].axis("off")

# LDA projection scatter
scores_all = lda_viz.decision_function(X_pca)
axes[3].hist(scores_all[y_all==0], bins=30, alpha=0.6, color=COLORS["nontarget"],
             label="non-target", density=True)
axes[3].hist(scores_all[y_all==1], bins=30, alpha=0.75, color=COLORS["target"],
             label="target", density=True)
axes[3].axvline(0, color=COLORS["green"], ls="--", lw=2, label="threshold=0")
axes[3].set_xlabel("LDA score")
axes[3].set_title("Score distribution\n(full data, visualization only)")
axes[3].legend(fontsize=8)

plt.suptitle("Fisherfaces — PCA+LDA visualization (full dataset, NOT used in CV)",
             fontsize=11, color=COLORS["gray"])
plt.tight_layout()
plt.show()

print(f"LDA shrinkage parameter (Ledoit-Wolf): {lda_viz.covariance_estimator_ if hasattr(lda_viz, 'covariance_estimator_') else 'auto'}")
print(f"PCA explained variance with 100 components: {pca_viz.explained_variance_ratio_.sum()*100:.1f}%")

## 2. PCA component sweep (on full data — for design choice)

How does the number of PCA components before LDA affect class separation?
We measure Fisher's criterion J = between-class / within-class scatter
as a proxy. The CV below uses n_pca=100 but this plot shows the landscape.

In [ ]:
n_pca_values = [10, 20, 30, 50, 75, 100, 120, 150]
fisher_scores = []

for n in n_pca_values:
    pca_tmp = PCA(n_components=n, random_state=67).fit_transform(X_s)
    lda_tmp = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    lda_tmp.fit(pca_tmp, y_all)
    scores_tmp = lda_tmp.decision_function(pca_tmp)
    # Fisher J = (mu_1 - mu_0)^2 / (sigma_1^2 + sigma_0^2)
    mu_t  = scores_tmp[y_all==1].mean()
    mu_nt = scores_tmp[y_all==0].mean()
    sig_t  = scores_tmp[y_all==1].std()
    sig_nt = scores_tmp[y_all==0].std()
    J = (mu_t - mu_nt)**2 / (sig_t**2 + sig_nt**2 + 1e-10)
    fisher_scores.append(J)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_pca_values, fisher_scores, "o-", color=COLORS["purple"], lw=2, ms=8)
ax.axvline(100, color=COLORS["green"], ls="--", lw=1.5, label="chosen: n_pca=100")
for n, j in zip(n_pca_values, fisher_scores):
    ax.annotate(f"{j:.1f}", (n, j), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=8)
ax.set_xlabel("PCA components before LDA")
ax.set_ylabel("Fisher criterion J (higher = better separation)")
ax.set_title("Effect of PCA dimensionality on LDA class separation\n(full data, design choice only — not CV)")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Cross-validation — strictly fold-aware

**Leakage audit per fold:**
- `StandardScaler`: fit on `X_train` only → `transform(X_val)`
- `PCA`: fit on `X_train_scaled` only → `transform(X_val_scaled)`
- `LDA`: fit on `X_train_pca` only → `decision_function(X_val_pca)`
- Val fold never touches fit: ✓

In [ ]:
N_PCA = 100
SEED  = 67

oof_scores   = np.full(len(manifest), np.nan)
fold_results = []
fold_val_data = []
fold_models  = []  # store for visualization

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    y_train  = train_df["label"].to_numpy()
    y_val    = val_df["label"].to_numpy()

    print(f"Fold {fold_id}: train={len(train_df)} "
          f"({y_train.sum()} target, {(y_train==0).sum()} non-target), "
          f"val={len(val_df)} ({y_val.sum()} target)")

    # Load
    X_train = load_images(train_df, DATA)
    X_val   = load_images(val_df,   DATA)

    # Step 1: StandardScaler (fit on train only)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    # Step 2: PCA (fit on train only)
    pca = PCA(n_components=N_PCA, random_state=SEED)
    X_train_pca = pca.fit_transform(X_train_s)
    X_val_pca   = pca.transform(X_val_s)

    # Step 3: LDA with Ledoit-Wolf shrinkage (fit on train only)
    lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    lda.fit(X_train_pca, y_train)

    # Score = decision_function (log-odds of target class)
    val_scores = lda.decision_function(X_val_pca)
    oof_scores[val_idx] = val_scores

    eer, _     = compute_eer(val_scores[y_val==1], val_scores[y_val==0])
    min_dcf, _ = compute_min_dcf(val_scores[y_val==1], val_scores[y_val==0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
    fold_val_data.append({"scores": val_scores, "labels": y_val})
    fold_models.append({"pca": pca, "lda": lda, "scaler": scaler})
    print(f"  → EER = {eer*100:.2f}%, min-DCF = {min_dcf:.4f}")

print("\nDone.")

## 4. Results

In [ ]:
results_df = pd.DataFrame(fold_results)
mean_eer   = results_df["eer"].mean()
std_eer    = results_df["eer"].std()
mean_dcf   = results_df["min_dcf"].mean()
std_dcf    = results_df["min_dcf"].std()

print("=" * 45)
print(f"{'Fold':<8} {'EER [%]':>10} {'min-DCF':>10}")
print("-" * 45)
for _, r in results_df.iterrows():
    print(f"{int(r.fold):<8} {r.eer*100:>10.2f} {r.min_dcf:>10.4f}")
print("-" * 45)
print(f"{'mean±std':<8} {mean_eer*100:>7.2f}±{std_eer*100:.2f} {mean_dcf:>7.4f}±{std_dcf:.4f}")
print("=" * 45)

eer_oof, _   = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_scores[y_all==1], oof_scores[y_all==0])
print(f"\nOOF overall: EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}, threshold = {thr:.3f}")
print(f"\nE004 PCA+logreg:    EER = 4.49 ± 4.26%,  min-DCF = 0.0565")
print(f"E006 PCA+LDA:       EER = {mean_eer*100:.2f} ± {std_eer*100:.2f}%, min-DCF = {mean_dcf:.4f}")
print(f"Change vs E004: {(mean_eer*100 - 4.49):+.2f}% EER")

## 5. Visualizations

In [ ]:
# Score distributions per fold + overall OOF
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
bin_edges = np.linspace(oof_scores.min(), oof_scores.max(), 35)

for i, (ax, fdata) in enumerate(zip(axes[:3], fold_val_data)):
    s, l = fdata["scores"], fdata["labels"]
    eer_f, thr_f = compute_eer(s[l==1], s[l==0])
    ax.hist(s[l==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
    ax.hist(s[l==1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
    ax.axvline(0,     color=COLORS["gray"],  ls=":",  lw=1.5, label="ideal thresh=0")
    ax.axvline(thr_f, color=COLORS["green"], ls="--", lw=2,   label=f"min-DCF thresh={thr_f:.2f}")
    ax.set_title(f"Fold {i}  —  EER = {eer_f*100:.1f}%")
    ax.set_xlabel("LDA score (log-odds)")
    ax.legend(fontsize=7)

ax = axes[3]
ax.hist(oof_scores[y_all==0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
ax.hist(oof_scores[y_all==1], bins=bin_edges, alpha=0.75, color=COLORS["target"],   label="target",     density=True)
ax.axvline(0,   color=COLORS["gray"],  ls=":",  lw=1.5, label="ideal thresh=0")
ax.axvline(thr, color=COLORS["green"], ls="--", lw=2,   label=f"OOF thresh={thr:.2f}")
ax.set_title(f"OOF overall  —  EER = {eer_oof*100:.1f}%")
ax.set_xlabel("LDA score (log-odds)")
ax.legend(fontsize=7)

plt.suptitle("E006 PCA+LDA — Score distributions per fold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fpr, tpr, _ = roc_curve(y_all, oof_scores)
roc_auc = auc(fpr, tpr)
frr_roc = 1 - tpr
eer_idx = np.argmin(np.abs(fpr - frr_roc))

ax = axes[0]
ax.plot(fpr, tpr, color=COLORS["target"], lw=2, label=f"AUC = {roc_auc:.3f}")
ax.plot([0,1],[0,1],"k--",lw=1,label="chance")
ax.scatter(fpr[eer_idx], tpr[eer_idx], color=COLORS["purple"], s=120, zorder=5,
           label=f"EER = {eer_oof*100:.1f}%")
ax.annotate(f"EER {eer_oof*100:.1f}%",
            xy=(fpr[eer_idx], tpr[eer_idx]),
            xytext=(fpr[eer_idx]+0.08, tpr[eer_idx]-0.1),
            arrowprops=dict(arrowstyle="->", color=COLORS["purple"]),
            color=COLORS["purple"], fontsize=9)
ax.set_xlabel("FAR"); ax.set_ylabel("1 − FRR")
ax.set_title("ROC (OOF)"); ax.legend()

ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t*100)}" for t in ticks]

ax = axes[1]
far_c = np.clip(fpr, 1e-4, 1-1e-4)
frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
        color=COLORS["target"], lw=2, label=f"E006 LDA  EER={eer_oof*100:.1f}%")
ax.plot(tick_pos, tick_pos, "k--", lw=1, label="EER line")
ax.scatter(scipy_norm.ppf(np.clip(fpr[eer_idx],1e-4,1-1e-4)),
           scipy_norm.ppf(np.clip(frr_roc[eer_idx],1e-4,1-1e-4)),
           color=COLORS["purple"], s=120, zorder=5)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]"); ax.set_ylabel("FRR [%]")
ax.set_title("DET (probit scale)"); ax.legend()

plt.suptitle("E006 PCA+LDA — ROC and DET curves", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Full image system progression: E004 vs E005 vs E006
experiments = {
    "E004\nPCA+logreg": {"fold_eers": [3.47, 0.83, 9.17], "mean": 4.49, "std": 4.26, "color": COLORS["nontarget"]},
    "E005\nLBP+logreg": {"fold_eers": [4.17, 4.17, 45.00], "mean": 17.78, "std": 23.58, "color": COLORS["gray"]},
    "E006\nPCA+LDA":    {"fold_eers": [r["eer"]*100 for r in fold_results],
                        "mean": mean_eer*100, "std": std_eer*100, "color": COLORS["target"]},
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Grouped bar chart per fold
ax = axes[0]
x = np.arange(3)
w = 0.25
for i, (name, data) in enumerate(experiments.items()):
    bars = ax.bar(x + (i-1)*w, data["fold_eers"], w,
                  label=name.replace("\n", " "), color=data["color"], alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        if h < 20:  # don't annotate the 45% bar to avoid clutter
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                    f"{h:.1f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(["Fold 0", "Fold 1", "Fold 2"])
ax.set_ylabel("EER [%]")
ax.set_title("Per-fold EER — image systems")
ax.legend(fontsize=8)

# Mean ± std comparison
ax = axes[1]
names = list(experiments.keys())
means = [d["mean"] for d in experiments.values()]
stds  = [d["std"]  for d in experiments.values()]
colors = [d["color"] for d in experiments.values()]
bars = ax.bar(range(len(names)), means, color=colors, alpha=0.85,
              yerr=stds, capsize=6, error_kw=dict(elinewidth=2))
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.5,
            f"{m:.1f}±{s:.1f}%", ha="center", fontsize=8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace("\n", "\n") for n in names], fontsize=9)
ax.set_ylabel("EER mean ± std [%]")
ax.set_title("Image system comparison — mean ± std")

plt.suptitle("Image modality progression: E004 → E005 → E006", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Fisherface direction per fold — shows how stable the discriminant is
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for fold_id, (ax, models) in enumerate(zip(axes, fold_models)):
    pca_f = models["pca"]
    lda_f = models["lda"]
    ff = (lda_f.coef_ @ pca_f.components_).reshape(80, 80)
    # Normalize for display
    ff_norm = (ff - ff.mean()) / (ff.std() + 1e-10)
    im = ax.imshow(ff_norm, cmap="RdBu_r", vmin=-3, vmax=3)
    ax.set_title(f"Fold {fold_id} Fisherface\n(val = session 0{fold_id+1})")
    ax.axis("off")

plt.colorbar(im, ax=axes.tolist(), label="Direction weight (red=target, blue=non-target)",
             shrink=0.8)
plt.suptitle("Fisherface per fold — how stable is the discriminant across sessions?",
             fontsize=11)
plt.tight_layout()
plt.show()